# KKU'3S Sentiment Analysis Project

## Project Introduction
This is a group course project supporting the **KKU'3S campus second-hand trading platform**.

The main purpose is to build a sentiment discrimination module to identify emotional tendencies (positive / neutral / negative) from platform user comments.



## 1. Check Runtime
Confirm GPU availability and record Python/PyTorch versions.

In [1]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


**Result:** Python 3.13.15, PyTorch 2.11.0+cu130, Tesla T4 available.

## 2. Load the Project
Clone the existing GitHub repository into Colab to reuse its files.

In [2]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/KKU_3S_Sentiment_Analysis")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/wuxunxun/KKU_3S_Sentiment_Analysis.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )

print("Project:", PROJECT_DIR)
subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=PROJECT_DIR,
    check=True,
)

Project: /content/KKU_3S_Sentiment_Analysis


CompletedProcess(args=['git', 'status', '--short', '--branch'], returncode=0)

**Result:** Project loaded successfully at /content/KKU_3S_Sentiment_Analysis.

## 3. Check Dependencies
Check the packages required for data loading, inference and evaluation.

In [4]:
import importlib

packages = [
    "torch", "datasets", "transformers", "pandas",
    "numpy", "sklearn", "matplotlib", "wandb", "emoji",
]

for name in packages:
    try:
        module = importlib.import_module(name)
        print(f"{name}: OK ({getattr(module, '__version__', 'unknown')})")
    except Exception as error:
        print(f"{name}: FAILED — {type(error).__name__}: {error}")

torch: OK (2.11.0+cu130)
datasets: OK (4.8.5)
transformers: OK (5.17.0)
pandas: OK (2.2.3)
numpy: OK (2.1.3)
sklearn: OK (1.6.1)
matplotlib: OK (3.10.0)
wandb: OK (0.28.1)
emoji: FAILED — ModuleNotFoundError: No module named 'emoji'


**Result:** Required packages are available; emoji 0.6.0 installed.

## 4. Load and Verify the Dataset
Check TweetEval sentiment splits, labels and class counts.

In [6]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

label_names = dataset["test"].features["label"].names
print("Labels:", label_names)

for split in ["train", "validation", "test"]:
    df = dataset[split].to_pandas()

    print(f"\n{split}: {len(df):,} samples")
    print(
        df["label"].value_counts()
        .sort_index()
        .rename(index=dict(enumerate(label_names)))
    )

test_df = dataset["test"].to_pandas()
test_df.insert(0, "sample_id", range(len(test_df)))

print("\nTest preview:")
display(test_df.head(3))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

...

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Labels: ['negative', 'neutral', 'positive']

train: 45,615 samples
label
negative     7093
neutral     20673
positive    17849
Name: count, dtype: int64

validation: 2,000 samples
label
negative    312
neutral     869
positive    819
Name: count, dtype: int64

test: 12,284 samples
label
negative    3972
neutral     5937
positive    2375
Name: count, dtype: int64

Test preview:


,sample_id,text,label
0,0,@user @user what do these '1/2 naked pics' hav...,1
1,1,OH: “I had a blue penis while I was this” [pla...,1
2,2,"@user @user That's coming, but I think the vic...",1


**Result:** TweetEval contains 45,615 training, 2,000 validation and
12,284 test samples. Labels are 0 = negative, 1 = neutral, 2 = positive.

The test set is imbalanced: neutral accounts for 48.3%, negative 32.3%,
and positive 19.3%. We will report Macro F1 and per-class metrics
alongside accuracy. All models will use the same test samples.

## 5. Define the Evaluation Setup

We choose the course option of **two new candidate models**.

| Role | Model |
|---|---|
| Baseline | cardiffnlp/twitter-roberta-base-sentiment-latest |
| Current production | LYTinn/gpt2-finetuning-sentiment-model-3000-samples |
| Candidate 1 | finiteautomata/bertweet-base-sentiment-analysis |
| Candidate 2 | cardiffnlp/twitter-xlm-roberta-base-sentiment |


**Settings:** Batch size = 16; maximum token length = 128.

In [9]:
MODEL_IDS = {
    "baseline": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "production": "LYTinn/gpt2-finetuning-sentiment-model-3000-samples",
    "bertweet": "finiteautomata/bertweet-base-sentiment-analysis",
    "xlm_roberta": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
}

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

BATCH_SIZE = 16
MAX_LENGTH = 128

for name, model_id in MODEL_IDS.items():
    print(f"{name}: {model_id}")

print("\nTest samples:", len(test_df))
print("Batch size:", BATCH_SIZE)
print("Maximum token length:", MAX_LENGTH)

baseline: cardiffnlp/twitter-roberta-base-sentiment-latest
production: LYTinn/gpt2-finetuning-sentiment-model-3000-samples
bertweet: finiteautomata/bertweet-base-sentiment-analysis
xlm_roberta: cardiffnlp/twitter-xlm-roberta-base-sentiment

Test samples: 12284
Batch size: 16
Maximum token length: 128


## 6. Check Test Data Quality
Check missing values, empty texts, duplicates and sample IDs before inference.

In [8]:
checks = {
    "Missing texts": int(test_df["text"].isna().sum()),
    "Missing labels": int(test_df["label"].isna().sum()),
    "Empty texts": int(
        test_df["text"].fillna("").str.strip().eq("").sum()
    ),
    "Duplicate texts": int(test_df["text"].duplicated().sum()),
    "Duplicate sample IDs": int(test_df["sample_id"].duplicated().sum()),
    "Invalid labels": int(~test_df["label"].isin(LABEL_MAP).all()),
}

for name, count in checks.items():
    print(f"{name}: {count}")

Missing texts: 0
Missing labels: 0
Empty texts: 0
Duplicate texts: 0
Duplicate sample IDs: 0
Invalid labels: 0


**Result:** All data-quality checks passed. All 12,284 test samples are retained.

## 7. Create Metadata
Create five text features for slice analysis: mentions, hashtags,
URLs, emojis and text length. These help compare model robustness
across different text characteristics.

In [10]:
import re
import pandas as pd

# Approximate emoji detection without the emoji package.
emoji_pattern = re.compile(
    "[\U0001F300-\U0001FAFF\u2600-\u27BF\U0001F1E6-\U0001F1FF]"
)

texts = test_df["text"]

test_df["mention_count"] = texts.str.count(r"@\w+")
test_df["hash_count"] = texts.str.count(r"#\w+")

test_df["has_url"] = texts.str.contains(
    r"https?://\S+|www\.\S+",
    regex=True,
    case=False,
    na=False,
)

test_df["has_emoji"] = texts.str.contains(
    emoji_pattern,
    regex=True,
    na=False,
)

# Raw-text character length: short <20, medium 20–100, long >100.
test_df["text_length_bin"] = pd.cut(
    texts.str.len(),
    bins=[-1, 19, 100, float("inf")],
    labels=["short", "medium", "long"],
)

metadata_cols = [
    "mention_count",
    "hash_count",
    "has_url",
    "has_emoji",
    "text_length_bin",
]

assert test_df[metadata_cols].notna().all().all(), "Missing metadata"
assert test_df["sample_id"].is_unique, "Duplicate sample IDs"

print("Samples:", len(test_df))
print("Missing metadata:", test_df[metadata_cols].isna().sum().sum())

print("\nFeature counts:")
print("With mentions:", (test_df["mention_count"] > 0).sum())
print("With hashtags:", (test_df["hash_count"] > 0).sum())
print("With URLs:", test_df["has_url"].sum())
print("With emojis:", test_df["has_emoji"].sum())

print("\nLength groups:")
print(
    test_df["text_length_bin"]
    .value_counts()
    .reindex(["short", "medium", "long"])
)

display(test_df[["sample_id", "text"] + metadata_cols].head(3))

Samples: 12284
Missing metadata: 0

Feature counts:
With mentions: 4997
With hashtags: 4825
With URLs: 376
With emojis: 814

Length groups:
text_length_bin
short       78
medium    6710
long      5496
Name: count, dtype: int64


,sample_id,text,mention_count,hash_count,has_url,has_emoji,text_length_bin
0,0,@user @user what do these '1/2 naked pics' hav...,2,0,False,False,medium
1,1,OH: “I had a blue penis while I was this” [pla...,0,0,False,False,medium
2,2,"@user @user That's coming, but I think the vic...",2,0,False,False,medium


## 8. Verify Model Labels
Check output classes and label names before defining model-specific mappings.

In [11]:
from transformers import AutoConfig

MODEL_CONFIGS = {}

for name, model_id in MODEL_IDS.items():
    config = AutoConfig.from_pretrained(model_id)
    MODEL_CONFIGS[name] = config

    print(f"\n{name}")
    print("Model type:", config.model_type)
    print("Number of labels:", config.num_labels)
    print("Label mapping:", config.id2label)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]


baseline
Model type: roberta
Number of labels: 3
Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}


config.json:   0%|          | 0.00/970 [00:00<?, ?B/s]


production
Model type: gpt2
Number of labels: 2
Label mapping: {0: 'LABEL_0', 1: 'LABEL_1'}


config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]


bertweet
Model type: roberta
Number of labels: 3
Label mapping: {0: 'NEG', 1: 'NEU', 2: 'POS'}


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]


xlm_roberta
Model type: xlm-roberta
Number of labels: 3
Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}


### Model Label Verification Results

| Role | Model ID | Classes | Native labels | Standard mapping | Verification |
|---|---|---:|---|---|---|
| Baseline | cardiffnlp/twitter-roberta-base-sentiment-latest | 3 | 0: negative; 1: neutral; 2: positive | 0→negative; 1→neutral; 2→positive | Explicit labels in loaded configuration |
| Production | LYTinn/gpt2-finetuning-sentiment-model-3000-samples | 2 | 0: LABEL_0; 1: LABEL_1 | Inferred: 0→negative; 1→positive | Inferred from IMDb labels; not explicitly confirmed by the author |
| Candidate 1 | finiteautomata/bertweet-base-sentiment-analysis | 3 | 0: NEG; 1: NEU; 2: POS | 0→negative; 1→neutral; 2→positive | Explicit labels in loaded configuration |
| Candidate 2 | cardiffnlp/twitter-xlm-roberta-base-sentiment | 3 | 0: negative; 1: neutral; 2: positive | 0→negative; 1→neutral; 2→positive | Explicit labels in loaded configuration |

#### Sources

- Baseline model:
  https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest
- Production model:
  https://huggingface.co/LYTinn/gpt2-finetuning-sentiment-model-3000-samples
- BERTweet model:
  https://huggingface.co/finiteautomata/bertweet-base-sentiment-analysis
- XLM-R model:
  https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment
- IMDb dataset label definitions:
  https://huggingface.co/datasets/stanfordnlp/imdb/blob/main/README.md



## 9. Test Baseline Inference
Run 5 samples to verify model loading, preprocessing and probability outputs.

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def normalize_tweet(text):
    tokens = []
    for token in text.split(" "):
        if token.startswith("@") and len(token) > 1:
            token = "@user"
        elif token.startswith("http"):
            token = "http"
        tokens.append(token)
    return " ".join(tokens)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

baseline_id = MODEL_IDS["baseline"]
baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_id)
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    baseline_id
).to(device)
baseline_model.eval()

preview = test_df[["sample_id", "text", "label"]].head(5).copy()
inputs = baseline_tokenizer(
    preview["text"].map(normalize_tweet).tolist(),
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    probabilities = torch.softmax(
        baseline_model(**inputs).logits, dim=-1
    ).cpu()

assert probabilities.shape == (5, 3)
assert torch.isfinite(probabilities).all()
assert torch.allclose(
    probabilities.sum(dim=1),
    torch.ones(5),
    atol=1e-5,
)

preview["ground_truth"] = preview["label"].map(LABEL_MAP)
preview["prediction"] = [
    baseline_model.config.id2label[int(i)]
    for i in probabilities.argmax(dim=1)
]
preview["confidence"] = probabilities.max(dim=1).values.numpy()

for i, label in baseline_model.config.id2label.items():
    preview[f"p_{label}"] = probabilities[:, int(i)].numpy()

print("Device:", device)
print("Probability checks passed.")
display(preview.drop(columns="label"))

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

Device: cuda
Probability checks passed.


,sample_id,text,ground_truth,prediction,confidence,p_negative,p_neutral,p_positive
0,0,@user @user what do these '1/2 naked pics' hav...,neutral,negative,0.804726,0.804726,0.187281,0.007993
1,1,OH: “I had a blue penis while I was this” [pla...,neutral,neutral,0.866949,0.041973,0.866949,0.091078
2,2,"@user @user That's coming, but I think the vic...",neutral,neutral,0.763724,0.203181,0.763724,0.033094
3,3,I think I may be finally in with the in crowd ...,positive,positive,0.774047,0.003712,0.222241,0.774047
4,4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,neutral,0.416397,0.303349,0.416397,0.280254


**Result:** 
All 5 samples produced valid 3-class probabilities.
This check confirms execution, not overall model quality.

## 10. Verify Production Inference
Check the binary production model on 40 balanced PN validation samples.
The provisional mapping is 0 = negative and 1 = positive.
This check is separate from final test evaluation.

In [17]:
production_id = MODEL_IDS["production"]

production_tokenizer = AutoTokenizer.from_pretrained(production_id)
production_model = AutoModelForSequenceClassification.from_pretrained(
    production_id
).to(device)

if production_tokenizer.pad_token_id is None:
    if production_tokenizer.eos_token_id is None:
        raise ValueError("No padding or EOS token.")
    production_tokenizer.pad_token = production_tokenizer.eos_token

production_model.config.pad_token_id = production_tokenizer.pad_token_id
production_model.eval()

val_df = dataset["validation"].to_pandas()
diagnostic_df = pd.concat([
    val_df[val_df["label"] == 0].head(20),
    val_df[val_df["label"] == 2].head(20),
]).copy()

prob_batches = []

with torch.inference_mode():
    for start in range(0, len(diagnostic_df), BATCH_SIZE):
        inputs = production_tokenizer(
            diagnostic_df["text"].iloc[
                start:start + BATCH_SIZE
            ].tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        prob_batches.append(
            torch.softmax(
                production_model(**inputs).logits, dim=-1
            ).cpu()
        )

production_probs = torch.cat(prob_batches)

assert production_probs.shape == (40, 2)
assert torch.isfinite(production_probs).all()
assert torch.allclose(
    production_probs.sum(dim=1), torch.ones(40), atol=1e-5
)

diagnostic_df["ground_truth"] = diagnostic_df["label"].map(LABEL_MAP)
diagnostic_df["predicted_id"] = production_probs.argmax(dim=1).numpy()

display(pd.crosstab(
    diagnostic_df["ground_truth"],
    diagnostic_df["predicted_id"],
).reindex(
    index=["negative", "positive"], columns=[0, 1], fill_value=0
))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: LYTinn/gpt2-finetuning-sentiment-model-3000-samples
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


predicted_id,0,1
ground_truth,,
negative,13,7
positive,7,13


## 11. Test BERTweet Inference
Install the missing emoji dependency before checking BERTweet on 5 samples.

In [18]:
import sys
import subprocess
import importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "emoji==0.6.0"],
    check=True,
)

importlib.invalidate_caches()
import emoji

print("emoji version:", emoji.__version__)
print("Conversion test:", emoji.demojize("😊", use_aliases=True))

emoji version: 0.6.0
Conversion test: :blush:


In [19]:
bertweet_id = MODEL_IDS["bertweet"]

bertweet_tokenizer = AutoTokenizer.from_pretrained(
    bertweet_id,
    normalization=True,
)
bertweet_model = AutoModelForSequenceClassification.from_pretrained(
    bertweet_id
).to(device)
bertweet_model.eval()

bertweet_preview = test_df[["sample_id", "text", "label"]].head(5).copy()

# BERTweet tokenizer performs its own text normalization.
inputs = bertweet_tokenizer(
    bertweet_preview["text"].tolist(),
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    bertweet_probs = torch.softmax(
        bertweet_model(**inputs).logits, dim=-1
    ).cpu()

assert bertweet_probs.shape == (5, 3)
assert torch.isfinite(bertweet_probs).all()
assert torch.allclose(
    bertweet_probs.sum(dim=1), torch.ones(5), atol=1e-5
)

bertweet_preview["ground_truth"] = bertweet_preview["label"].map(LABEL_MAP)
bertweet_preview["prediction"] = [
    LABEL_MAP[int(i)] for i in bertweet_probs.argmax(dim=1)
]
bertweet_preview["confidence"] = (
    bertweet_probs.max(dim=1).values.numpy()
)

for i, label in LABEL_MAP.items():
    bertweet_preview[f"p_{label}"] = bertweet_probs[:, i].numpy()

print("BERTweet probability checks passed.")
display(bertweet_preview.drop(columns="label"))

tokenizer_config.json:   0%|          | 0.00/338 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERTweet probability checks passed.


,sample_id,text,ground_truth,prediction,confidence,p_negative,p_neutral,p_positive
0,0,@user @user what do these '1/2 naked pics' hav...,neutral,negative,0.702583,0.702583,0.293003,0.004414
1,1,OH: “I had a blue penis while I was this” [pla...,neutral,neutral,0.917466,0.010324,0.917466,0.072210
2,2,"@user @user That's coming, but I think the vic...",neutral,negative,0.823702,0.823702,0.172655,0.003643
3,3,I think I may be finally in with the in crowd ...,positive,positive,0.821122,0.001694,0.177184,0.821122
4,4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,negative,0.756727,0.756727,0.214951,0.028321


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

## 12. Test XLM-R Inference
Run the same five samples using mention/URL normalization and
verify three-class probability outputs.

In [20]:
xlm_id = MODEL_IDS["xlm_roberta"]

xlm_tokenizer = AutoTokenizer.from_pretrained(xlm_id)
xlm_model = AutoModelForSequenceClassification.from_pretrained(
    xlm_id
).to(device)
xlm_model.eval()

xlm_preview = test_df[["sample_id", "text", "label"]].head(5).copy()

inputs = xlm_tokenizer(
    xlm_preview["text"].map(normalize_tweet).tolist(),
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    xlm_probs = torch.softmax(
        xlm_model(**inputs).logits, dim=-1
    ).cpu()

assert xlm_probs.shape == (5, 3)
assert torch.isfinite(xlm_probs).all()
assert torch.allclose(
    xlm_probs.sum(dim=1), torch.ones(5), atol=1e-5
)

xlm_preview["ground_truth"] = xlm_preview["label"].map(LABEL_MAP)
xlm_preview["prediction"] = [
    xlm_model.config.id2label[int(i)]
    for i in xlm_probs.argmax(dim=1)
]
xlm_preview["confidence"] = xlm_probs.max(dim=1).values.numpy()

for i, label in xlm_model.config.id2label.items():
    xlm_preview[f"p_{label}"] = xlm_probs[:, int(i)].numpy()

print("XLM-R probability checks passed.")
display(xlm_preview.drop(columns="label"))

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLM-R probability checks passed.


,sample_id,text,ground_truth,prediction,confidence,p_negative,p_neutral,p_positive
0,0,@user @user what do these '1/2 naked pics' hav...,neutral,negative,0.578722,0.578722,0.397967,0.023311
1,1,OH: “I had a blue penis while I was this” [pla...,neutral,neutral,0.825280,0.076268,0.825280,0.098452
2,2,"@user @user That's coming, but I think the vic...",neutral,neutral,0.614876,0.330814,0.614876,0.054310
3,3,I think I may be finally in with the in crowd ...,positive,positive,0.537997,0.073295,0.388708,0.537997
4,4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,negative,0.786067,0.786067,0.148329,0.065604


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

## 13. Run Full Baseline Inference
Run all 12,284 test samples and save native predictions, confidence
and three-class probabilities. PN-adapted predictions will be derived
later, separately from native predictions.

In [21]:
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_DIR = PROJECT_DIR / "results" / "final_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

baseline_results = test_df.copy()
baseline_results["text_input"] = (
    baseline_results["text"].map(normalize_tweet)
)

prob_batches = []
baseline_model.eval()

with torch.inference_mode():
    for start in tqdm(
        range(0, len(baseline_results), BATCH_SIZE),
        desc="Baseline inference",
    ):
        batch = baseline_results["text_input"].iloc[
            start:start + BATCH_SIZE
        ].tolist()

        inputs = baseline_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        prob_batches.append(
            torch.softmax(
                baseline_model(**inputs).logits, dim=-1
            ).cpu()
        )

baseline_probs = torch.cat(prob_batches)

assert baseline_probs.shape == (len(test_df), 3)
assert torch.isfinite(baseline_probs).all()
assert torch.allclose(
    baseline_probs.sum(dim=1),
    torch.ones(len(test_df)),
    atol=1e-5,
)

baseline_results["model_id"] = MODEL_IDS["baseline"]
baseline_results["ground_truth"] = baseline_results["label"].map(LABEL_MAP)
baseline_results["pred_native_id"] = baseline_probs.argmax(dim=1).numpy()
baseline_results["prediction_native"] = (
    baseline_results["pred_native_id"].map(LABEL_MAP)
)
baseline_results["confidence_native"] = (
    baseline_probs.max(dim=1).values.numpy()
)

for i, label in LABEL_MAP.items():
    baseline_results[f"p_{label}"] = baseline_probs[:, i].numpy()

output_path = OUTPUT_DIR / "baseline_predictions.csv"
baseline_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(baseline_results))
print("Unique sample IDs:", baseline_results["sample_id"].nunique())
display(baseline_results.head(3))

Baseline inference:   0%|          | 0/768 [00:00<?, ?it/s]

Saved: /content/KKU_3S_Sentiment_Analysis/results/final_run/baseline_predictions.csv
Rows: 12284
Unique sample IDs: 12284


,sample_id,text,label,mention_count,hash_count,has_url,has_emoji,text_length_bin,text_input,model_id,ground_truth,pred_native_id,prediction_native,confidence_native,p_negative,p_neutral,p_positive
0,0,@user @user what do these '1/2 naked pics' hav...,1,2,0,False,False,medium,@user @user what do these '1/2 naked pics' hav...,cardiffnlp/twitter-roberta-base-sentiment-latest,neutral,0,negative,0.804726,0.804726,0.187281,0.007993
1,1,OH: “I had a blue penis while I was this” [pla...,1,0,0,False,False,medium,OH: “I had a blue penis while I was this” [pla...,cardiffnlp/twitter-roberta-base-sentiment-latest,neutral,1,neutral,0.866949,0.041973,0.866949,0.091078
2,2,"@user @user That's coming, but I think the vic...",1,2,0,False,False,medium,"@user @user That's coming, but I think the vic...",cardiffnlp/twitter-roberta-base-sentiment-latest,neutral,1,neutral,0.763724,0.203181,0.763724,0.033094


## 14. Run Full Production Inference
Save binary outputs for all test samples using the repository tokenizer.
PN evaluation will use only ground-truth negative/positive samples.
The label mapping remains provisional: 0 = negative, 1 = positive.

In [22]:
production_results = test_df.copy()
production_results["text_input"] = production_results["text"]

prob_batches = []
production_model.eval()

with torch.inference_mode():
    for start in tqdm(
        range(0, len(production_results), BATCH_SIZE),
        desc="Production inference",
    ):
        inputs = production_tokenizer(
            production_results["text_input"].iloc[
                start:start + BATCH_SIZE
            ].tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        prob_batches.append(
            torch.softmax(
                production_model(**inputs).logits, dim=-1
            ).cpu()
        )

production_probs = torch.cat(prob_batches)

assert production_probs.shape == (len(test_df), 2)
assert torch.isfinite(production_probs).all()
assert torch.allclose(
    production_probs.sum(dim=1),
    torch.ones(len(test_df)),
    atol=1e-5,
)

production_results["model_id"] = MODEL_IDS["production"]
production_results["ground_truth"] = (
    production_results["label"].map(LABEL_MAP)
)
production_results["pred_native_id"] = (
    production_probs.argmax(dim=1).numpy()
)
production_results["prediction_native"] = (
    production_results["pred_native_id"].map(
        {0: "negative", 1: "positive"}
    )
)
production_results["confidence_native"] = (
    production_probs.max(dim=1).values.numpy()
)
production_results["p_negative"] = production_probs[:, 0].numpy()
production_results["p_positive"] = production_probs[:, 1].numpy()
production_results["label_mapping_status"] = "provisional_imdb_based"

output_path = OUTPUT_DIR / "production_predictions.csv"
production_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(production_results))
print("Unique sample IDs:", production_results["sample_id"].nunique())
print("PN evaluation rows:",
      production_results["label"].isin([0, 2]).sum())

Production inference:   0%|          | 0/768 [00:00<?, ?it/s]

Saved: /content/KKU_3S_Sentiment_Analysis/results/final_run/production_predictions.csv
Rows: 12284
Unique sample IDs: 12284
PN evaluation rows: 6347


## 15. Run Full BERTweet Inference
Run all test samples with BERTweet tokenizer normalization enabled.
Save native three-class predictions and probabilities for later evaluation.

In [23]:
bertweet_results = test_df.copy()

# Record the tokenizer-normalized text used for inference.
bertweet_results["text_input"] = bertweet_results["text"].map(
    bertweet_tokenizer.normalizeTweet
)

prob_batches = []
bertweet_model.eval()

with torch.inference_mode():
    for start in tqdm(
        range(0, len(bertweet_results), BATCH_SIZE),
        desc="BERTweet inference",
    ):
        # Pass raw text; the tokenizer applies normalization internally.
        inputs = bertweet_tokenizer(
            bertweet_results["text"].iloc[
                start:start + BATCH_SIZE
            ].tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        prob_batches.append(
            torch.softmax(
                bertweet_model(**inputs).logits, dim=-1
            ).cpu()
        )

bertweet_probs = torch.cat(prob_batches)

assert bertweet_probs.shape == (len(test_df), 3)
assert torch.isfinite(bertweet_probs).all()
assert torch.allclose(
    bertweet_probs.sum(dim=1),
    torch.ones(len(test_df)),
    atol=1e-5,
)

bertweet_results["model_id"] = MODEL_IDS["bertweet"]
bertweet_results["ground_truth"] = bertweet_results["label"].map(LABEL_MAP)
bertweet_results["pred_native_id"] = bertweet_probs.argmax(dim=1).numpy()
bertweet_results["prediction_native"] = (
    bertweet_results["pred_native_id"].map(LABEL_MAP)
)
bertweet_results["confidence_native"] = (
    bertweet_probs.max(dim=1).values.numpy()
)

for i, label in LABEL_MAP.items():
    bertweet_results[f"p_{label}"] = bertweet_probs[:, i].numpy()

output_path = OUTPUT_DIR / "bertweet_predictions.csv"
bertweet_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(bertweet_results))
print("Unique sample IDs:", bertweet_results["sample_id"].nunique())

BERTweet inference:   0%|          | 0/768 [00:00<?, ?it/s]

Saved: /content/KKU_3S_Sentiment_Analysis/results/final_run/bertweet_predictions.csv
Rows: 12284
Unique sample IDs: 12284


## 16. Run Full XLM-R Inference
Run all test samples with mention/URL normalization.
Save native three-class predictions and probabilities for later evaluation.

In [24]:
xlm_results = test_df.copy()
xlm_results["text_input"] = xlm_results["text"].map(normalize_tweet)

prob_batches = []
xlm_model.eval()

with torch.inference_mode():
    for start in tqdm(
        range(0, len(xlm_results), BATCH_SIZE),
        desc="XLM-R inference",
    ):
        inputs = xlm_tokenizer(
            xlm_results["text_input"].iloc[
                start:start + BATCH_SIZE
            ].tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        prob_batches.append(
            torch.softmax(
                xlm_model(**inputs).logits, dim=-1
            ).cpu()
        )

xlm_probs = torch.cat(prob_batches)

assert xlm_probs.shape == (len(test_df), 3)
assert torch.isfinite(xlm_probs).all()
assert torch.allclose(
    xlm_probs.sum(dim=1),
    torch.ones(len(test_df)),
    atol=1e-5,
)

xlm_results["model_id"] = MODEL_IDS["xlm_roberta"]
xlm_results["ground_truth"] = xlm_results["label"].map(LABEL_MAP)
xlm_results["pred_native_id"] = xlm_probs.argmax(dim=1).numpy()
xlm_results["prediction_native"] = (
    xlm_results["pred_native_id"].map(LABEL_MAP)
)
xlm_results["confidence_native"] = (
    xlm_probs.max(dim=1).values.numpy()
)

for i, label in LABEL_MAP.items():
    xlm_results[f"p_{label}"] = xlm_probs[:, i].numpy()

output_path = OUTPUT_DIR / "xlm_roberta_predictions.csv"
xlm_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(xlm_results))
print("Unique sample IDs:", xlm_results["sample_id"].nunique())

XLM-R inference:   0%|          | 0/768 [00:00<?, ?it/s]

Saved: /content/KKU_3S_Sentiment_Analysis/results/final_run/xlm_roberta_predictions.csv
Rows: 12284
Unique sample IDs: 12284


## 17. Verify Prediction Files
Check sample alignment, labels and probabilities across all four
saved files before computing metrics.

In [25]:
import numpy as np

prediction_tables = {}
check_rows = []

reference = test_df.set_index("sample_id").sort_index()

for name in MODEL_IDS:
    path = OUTPUT_DIR / f"{name}_predictions.csv"
    df = pd.read_csv(path)

    assert len(df) == len(reference), f"{name}: incorrect row count"
    assert df["sample_id"].is_unique, f"{name}: duplicate IDs"

    df = df.set_index("sample_id").sort_index()

    assert df.index.equals(reference.index), f"{name}: IDs differ"
    assert df["text"].equals(reference["text"]), f"{name}: texts differ"
    assert df["label"].equals(reference["label"]), f"{name}: labels differ"
    assert df["model_id"].eq(MODEL_IDS[name]).all(), f"{name}: model ID differs"
    assert df["ground_truth"].equals(
        reference["label"].map(LABEL_MAP)
    ), f"{name}: ground-truth mapping differs"

    if name == "production":
        prob_cols = ["p_negative", "p_positive"]
        native_map = {0: "negative", 1: "positive"}
    else:
        prob_cols = ["p_negative", "p_neutral", "p_positive"]
        native_map = LABEL_MAP

    probs = df[prob_cols].to_numpy()

    assert np.isfinite(probs).all(), f"{name}: invalid probabilities"
    assert ((probs >= 0) & (probs <= 1)).all()
    assert np.allclose(probs.sum(axis=1), 1, atol=1e-5)
    assert np.array_equal(
        probs.argmax(axis=1), df["pred_native_id"].to_numpy()
    ), f"{name}: prediction/probability mismatch"
    assert df["prediction_native"].equals(
        df["pred_native_id"].map(native_map)
    ), f"{name}: prediction mapping differs"
    assert np.allclose(
        probs.max(axis=1), df["confidence_native"], atol=1e-5
    ), f"{name}: confidence mismatch"

    for col in metadata_cols:
        assert df[col].equals(reference[col].astype(df[col].dtype)), (
            f"{name}: metadata mismatch in {col}"
        )

    prediction_tables[name] = df.reset_index()
    check_rows.append({
        "model": name,
        "rows": len(df),
        "PN_rows": int(df["label"].isin([0, 2]).sum()),
        "status": "Passed",
    })

display(pd.DataFrame(check_rows))

,model,rows,PN_rows,status
0,baseline,12284,6347,Passed
1,production,12284,6347,Passed
2,bertweet,12284,6347,Passed
3,xlm_roberta,12284,6347,Passed


**Result:** All four prediction files passed verification.
Each contains the same 12,284 samples, including 6,347 PN samples.

## 18. Evaluate PN Performance
Evaluate all four models on the same 6,347 ground-truth PN samples.
Three-class models use argmax(p_negative, p_positive) for this
adapted binary task. Native predictions remain unchanged.
Production label semantics remain provisional.

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

pn_metric_rows = []
pn_tables = {}
reference_pn_ids = None

for name, df in prediction_tables.items():
    # Filter by ground truth only.
    pn = df[df["label"].isin([0, 2])].copy()

    current_ids = pn["sample_id"].to_numpy()
    if reference_pn_ids is None:
        reference_pn_ids = current_ids
    else:
        assert np.array_equal(current_ids, reference_pn_ids)

    assert len(pn) == 6347

    # Unified PN encoding: 0 = negative, 1 = positive.
    y_true = pn["label"].map({0: 0, 2: 1}).to_numpy()
    pn_probs = pn[["p_negative", "p_positive"]].to_numpy()
    y_pred = pn_probs.argmax(axis=1)

    # Conditional PN confidence for three-class models.
    pn_mass = pn_probs.sum(axis=1)
    assert (pn_mass > 0).all()

    pn["pred_pn_id"] = y_pred
    pn["prediction_pn"] = pd.Series(
        y_pred, index=pn.index
    ).map({0: "negative", 1: "positive"})
    pn["confidence_pn"] = pn_probs.max(axis=1) / pn_mass
    pn["evaluation_protocol"] = (
        "native_binary" if name == "production"
        else "adapted_binary_ignore_neutral"
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    pn_metric_rows.append({
        "model": name,
        "samples": len(pn),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1.mean(),
        "negative_precision": precision[0],
        "negative_recall": recall[0],
        "negative_f1": f1[0],
        "positive_precision": precision[1],
        "positive_recall": recall[1],
        "positive_f1": f1[1],
    })

    pn_tables[name] = pn
    pn.to_csv(OUTPUT_DIR / f"{name}_pn_predictions.csv", index=False)

pn_metrics = pd.DataFrame(pn_metric_rows)
pn_metrics.to_csv(OUTPUT_DIR / "metrics_pn.csv", index=False)

display(pn_metrics.round(4))
print("Saved PN predictions and metrics.")

,model,samples,accuracy,macro_f1,negative_precision,negative_recall,negative_f1,positive_precision,positive_recall,positive_f1
0,baseline,6347,0.9464,0.9430,0.9609,0.9532,0.9570,0.9227,0.9352,0.9289
1,production,6347,0.5360,0.5261,0.6559,0.5438,0.5946,0.4067,0.5229,0.4575
2,bertweet,6347,0.9460,0.9429,0.9717,0.9411,0.9561,0.9064,0.9541,0.9296
3,xlm_roberta,6347,0.9263,0.9208,0.9339,0.9494,0.9416,0.9129,0.8876,0.9001


Saved PN predictions and metrics.


**Conclusion:** On the same 6,347 PN samples, BERTweet and XLM-R
achieved higher scores than production. BERTweet's aggregate scores
were close to baseline, while XLM-R had higher negative recall than
BERTweet but lower Macro F1.

These results evaluate PN-adapted performance, not native three-class
performance. Production results remain subject to the documented
label-mapping and tokenizer uncertainties. No shipping decision is
made at this stage.

## 19. Evaluate Native Three-Class Performance
Evaluate baseline and both candidates on all 12,284 test samples.
Use native predictions and fixed negative/neutral/positive labels.
Production is excluded from this three-class comparison.

In [27]:
three_class_rows = []

for name in ["baseline", "bertweet", "xlm_roberta"]:
    df = prediction_tables[name]

    y_true = df["label"].to_numpy()
    y_pred = df["pred_native_id"].to_numpy()

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

    row = {
        "model": name,
        "samples": len(df),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
    }

    for i, label in LABEL_MAP.items():
        row[f"{label}_precision"] = precision[i]
        row[f"{label}_recall"] = recall[i]
        row[f"{label}_f1"] = f1[i]
        row[f"{label}_support"] = int(support[i])

    three_class_rows.append(row)

three_class_metrics = pd.DataFrame(three_class_rows)
three_class_metrics.to_csv(
    OUTPUT_DIR / "metrics_three_class.csv",
    index=False,
)

display(three_class_metrics[[
    "model", "samples", "accuracy", "macro_f1", "macro_recall",
    "negative_recall", "negative_f1", "neutral_f1", "positive_f1",
]].round(4))

,model,samples,accuracy,macro_f1,macro_recall,negative_recall,negative_f1,neutral_f1,positive_f1
0,baseline,12284,0.7218,0.7240,0.7349,0.8066,0.7435,0.7036,0.7251
1,bertweet,12284,0.7202,0.7219,0.7364,0.8021,0.7503,0.6987,0.7168
2,xlm_roberta,12284,0.6819,0.6842,0.7061,0.8693,0.7204,0.6416,0.6908


**Conclusion:** BERTweet performed close to baseline on the full
three-class test set, without an aggregate improvement in Macro F1.

XLM-R achieved the highest negative recall (0.8693), but lower neutral
F1 (0.6416) and Macro F1 (0.6842), indicating a trade-off between
negative detection and balanced classification.

BERTweet meets the provisional 0.70 thresholds for accuracy, Macro
Recall and Macro F1. XLM-R does not meet the accuracy and Macro F1
thresholds. Slice analysis and error inspection are still required
before making a shipping decision.

## 20. Analyze Metadata Slices
Compare performance across the five provisional metadata features.
Keep PN-adapted and native three-class results separate.
Report sample counts and class support to identify small slices.

In [29]:
slice_rules = {
    "mention_count": [
        ("none", lambda df: df["mention_count"] == 0),
        ("present", lambda df: df["mention_count"] > 0),
    ],
    "hash_count": [
        ("none", lambda df: df["hash_count"] == 0),
        ("present", lambda df: df["hash_count"] > 0),
    ],
    "has_url": [
        ("no", lambda df: df["has_url"] == False),
        ("yes", lambda df: df["has_url"] == True),
    ],
    "has_emoji": [
        ("no", lambda df: df["has_emoji"] == False),
        ("yes", lambda df: df["has_emoji"] == True),
    ],
    "text_length_bin": [
        (group, lambda df, g=group: df["text_length_bin"] == g)
        for group in ["short", "medium", "long"]
    ],
}

def compute_slice_metrics(tables, protocol):
    rows = []
    class_ids = [0, 1] if protocol == "PN" else [0, 1, 2]

    for model_name, df in tables.items():
        for feature, groups in slice_rules.items():
            for group_name, select in groups:
                part = df[select(df)]
                if part.empty:
                    continue

                if protocol == "PN":
                    y_true = part["label"].map({0: 0, 2: 1}).to_numpy()
                    y_pred = part["pred_pn_id"].to_numpy()
                else:
                    y_true = part["label"].to_numpy()
                    y_pred = part["pred_native_id"].to_numpy()

                _, recall, f1, support = precision_recall_fscore_support(
                    y_true,
                    y_pred,
                    labels=class_ids,
                    average=None,
                    zero_division=0,
                )

                # Missing true classes make fixed-class Macro F1
                # unsuitable for this slice; mark it as unavailable.
                all_classes_present = (support > 0).all()

                rows.append({
                    "protocol": protocol,
                    "model": model_name,
                    "metadata": feature,
                    "slice": group_name,
                    "samples": len(part),
                    "negative_support": int(support[0]),
                    "neutral_support": (
                        int(support[1]) if protocol != "PN" else np.nan
                    ),
                    "positive_support": int(support[-1]),
                    "accuracy": accuracy_score(y_true, y_pred),
                    "macro_f1": (
                        f1.mean() if all_classes_present else np.nan
                    ),
                    "negative_recall": (
                        recall[0] if support[0] > 0 else np.nan
                    ),
                    "small_slice": (
                        len(part) < 100 or support.min() < 20
                    ),
                })

    return pd.DataFrame(rows)

pn_slices = compute_slice_metrics(pn_tables, "PN")

three_class_tables = {
    name: prediction_tables[name]
    for name in ["baseline", "bertweet", "xlm_roberta"]
}
three_class_slices = compute_slice_metrics(
    three_class_tables, "three_class"
)

pn_slices.to_csv(OUTPUT_DIR / "metrics_slices_pn.csv", index=False)
three_class_slices.to_csv(
    OUTPUT_DIR / "metrics_slices_three_class.csv", index=False
)

for title, results in [
    ("PN", pn_slices),
    ("Three-class", three_class_slices),
]:
    print(f"\n{title}")
    print("Models:", results["model"].nunique())
    print("Slices per model:",
          results[["metadata", "slice"]].drop_duplicates().shape[0])
    print("Metric records:", len(results))
    print("Small-sample slices per model:",
          results.groupby("model")["small_slice"].sum().to_dict())
    print("Records without Macro F1:",
          results["macro_f1"].isna().sum())

    for name, group in results.groupby("model"):
        usable = group[
            ~group["small_slice"] & group["macro_f1"].notna()
        ]

        if usable.empty:
            print(f"{name}: no sufficiently supported slices")
            continue

        lowest = usable.loc[usable["macro_f1"].idxmin()]
        print(
            f"{name}: lowest supported slice Macro F1 "
            f"= {lowest['macro_f1']:.4f} "
            f"({lowest['metadata']}={lowest['slice']}, "
            f"N={int(lowest['samples'])})"
        )

print("\nSaved: metrics_slices_pn.csv")
print("Saved: metrics_slices_three_class.csv")


PN
Models: 4
Slices per model: 11
Metric records: 44
Small-sample slices per model: {'baseline': 1, 'bertweet': 1, 'production': 1, 'xlm_roberta': 1}
Records without Macro F1: 0
baseline: lowest supported slice Macro F1 = 0.9243 (has_url=yes, N=129)
bertweet: lowest supported slice Macro F1 = 0.9287 (text_length_bin=long, N=3179)
production: lowest supported slice Macro F1 = 0.4872 (has_emoji=yes, N=544)
xlm_roberta: lowest supported slice Macro F1 = 0.9005 (has_url=yes, N=129)

Three-class
Models: 3
Slices per model: 11
Metric records: 33
Small-sample slices per model: {'baseline': 1, 'bertweet': 1, 'xlm_roberta': 1}
Records without Macro F1: 0
baseline: lowest supported slice Macro F1 = 0.6468 (has_url=yes, N=376)
bertweet: lowest supported slice Macro F1 = 0.6340 (has_url=yes, N=376)
xlm_roberta: lowest supported slice Macro F1 = 0.6567 (has_url=yes, N=376)

Saved: metrics_slices_pn.csv
Saved: metrics_slices_three_class.csv
